# 09_2. Effect Size Calculation (Cohen's d & Hedges' g)

Interactive Jupyter Notebook analysis script.

### 1. Import Libraries


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def cohens_d(x: np.ndarray, y: np.ndarray) -> float:
    nx, ny = len(x), len(y)
    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)
    pooled_sd = np.sqrt(((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2))
    return float((np.mean(x) - np.mean(y)) / pooled_sd)


def hedges_g(x: np.ndarray, y: np.ndarray) -> float:
    d = cohens_d(x, y)
    df = len(x) + len(y) - 2
    j = 1 - (3 / (4 * df - 1))
    return float(d * j)


def plot_effect_sizes(effect_dict: dict[str, float], outpath: Path) -> None:
    # Sort effect sizes descending (max to min)
    sorted_items = sorted(effect_dict.items(), key=lambda item: item[1], reverse=False)
    names = [k for k, _ in sorted_items]
    values = [v for _, v in sorted_items]
    colors = ["#2C7FB8" if v >= 0 else "#D9534F" for v in values]

    fig, ax = plt.subplots(figsize=(11, 5.5))
    bars = ax.barh(names, values, color=colors, alpha=0.85, edgecolor="black", height=0.55)
    
    ax.axvline(0, color="black", linestyle="-", lw=1.2)
    ax.axvline(0.2, color="gray", linestyle=":", label="Small (0.2)")
    ax.axvline(0.5, color="orange", linestyle="--", label="Medium (0.5)")
    ax.axvline(0.8, color="red", linestyle="-.", label="Large (0.8)")

    for bar in bars:
        w = bar.get_width()
        xpos = w + (0.06 if w >= 0 else -0.16)
        ax.text(xpos, bar.get_y() + bar.get_height()/2.0, f"{w:.2f}", va="center", fontweight="bold", fontsize=9.5)

    max_val = max(abs(min(values)), abs(max(values)))
    ax.set_xlim(-max_val - 0.5, max_val + 0.6)

    ax.set_xlabel("Effect Size (Cohen's d / Hedges' g)", fontweight="bold", fontsize=11)
    ax.set_title("Biomarker Effect Size Magnitudes (Sorted Max to Min)", fontweight="bold", fontsize=12)
    ax.legend(loc="lower right", frameon=True, framealpha=0.9)
    ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    outpath.parent.mkdir(exist_ok=True, parents=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    np.random.seed(42)
    g_ctrl = np.random.normal(10, 1.2, 30)
    g_ta = np.random.normal(11.8, 1.1, 30)
    g_tb = np.random.normal(9.0, 1.1, 30)

    effects = {
        "Treatment_A vs Control (Cohen d)": cohens_d(g_ta, g_ctrl),
        "Treatment_A vs Control (Hedges g)": hedges_g(g_ta, g_ctrl),
        "Treatment_B vs Control (Cohen d)": cohens_d(g_tb, g_ctrl),
        "Treatment_B vs Control (Hedges g)": hedges_g(g_tb, g_ctrl),
    }
    for k, v in effects.items():
        print(f"  {k:<35}: {v:.4f}")

    out_dir = Path(__file__).parent
    outpath = out_dir / "09_2_effect_size_analysis.png"
    plot_effect_sizes(effects, outpath)
    print(f"Effect size plot saved to: {outpath.resolve()}")


if __name__ == "__main__":
    main()


### 2. Compute Standardized Effect Sizes


In [ ]:
np.random.seed(42)
g1 = np.random.normal(10, 1.2, 25)
g2 = np.random.normal(11.8, 1.3, 25)
effects = {'T_A vs Control (d)': cohens_d(g2, g1), 'T_A vs Control (g)': hedges_g(g2, g1)}
print(effects)


### 3. Visualize Magnitude Thresholds


In [ ]:
outpath = Path('09_2_effect_size_analysis.png')
plot_effect_sizes(effects, outpath)
plt.show()
